# Suite PYS — Python3 simplified by Python's rules

`Python3.Simplifiers.OfAny(term, scope)` simplifies a Python3 term by Python's rules, CPython 3.14's (mbse-basis
[CAP-295: simplification of single-dialect expressions with literal leaves](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-295) to
[CAP-297: simplification in every binding state](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-297)): Python runs
them, and TypeScript models them over JavaScript's values, the same tables of values and errors in both. What the scope
decides folds into the literal of its value (a basic scalar's, or a constant's: `None`, `...`, a complex number), a
list, a tuple, a dict or a non-empty set of such values into its display, a closed constructor expression; a value with
none (an empty set, a slice, a closure, a template, a generator) keeps its subterm, its own subterms simplified; and what Python leaves to a statement, to its
implementation or to its platform stays, named with its class in the evaluation inventory (`residues`): `yield` and
`await` (`d`), `1 is 1` (`b`), a complex power or a locale's format (`a`). What Python makes an error still raises,
by the same class and message. Terms are built through the dialect's writers, in a transaction (`term`); `text` gives
a residual's source, as the printer writes it.

The scope's options choose otherwise for what Python leaves to its implementation or its platform (mbse-basis
[CAP-305: simplification's options, chosen up front](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-305.md),
PYS-15 to PYS-19): a refused format spec raises, equal immutable values are compared as CPython compares them, a
complex power folds by the host's math library, a locale's spec is formatted under a locale given as data, and a field
whose text is the implementation's is written as CPython writes it.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import { Errors as FE, Terms as F } from "@mbse/expressions/Framework";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
import { Builders, Errors, Repr } from "@mbse/schemas/Framework";
import { readFileSync } from "node:fs";
import { assert, raises } from "./support.js";

const { NotEvaluated } = Simplification;
const { ValueError } = Errors;
const { IndexError, OverflowError, ZeroDivisionError } = FE;
const { Tuple } = Repr;
const [P, PD, PV, PText] = [Python3.Expressions, Python3.Domains, Python3.Simplifiers, Python3.Text];

const store: any = new Builders.Store(); // the framework, the standard dialects Python3's complex numbers use, and the Python family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
  PythonTypes.install, PythonPatterns.install]) {
  store.transaction(install);
}

type Variables = Record<string, unknown>;

/** The Python3 term `spec` writes, a native value as its literal, the root of a holder built within the transaction
 * open: the term, a visitor of that transaction, lives within it. */
function term(spec: unknown): any {
  return F.term(store.Expressions.Framework.Expression((e: any) => e.root(P.DIALECT.spec(spec))));
}

/** The source of the term `spec` writes. */
const source = (spec: unknown): string => PText.ToText(term(spec));

/** The simplification of the term `spec` writes, in a scope binding `variables`. */
const simplified = (spec: unknown, variables: Variables = {}) => PV.OfAny(term(spec), new PV.Scope(variables));

/** The value the term `spec` writes simplifies to: its literal root's. */
const value = (spec: unknown, variables: Variables = {}): unknown => simplified(spec, variables).evaluate();

/** The source of the residual of the term `spec` writes, simplified in a scope binding `variables`. */
const text = (spec: unknown, variables: Variables = {}): string => PText.ToText(term(simplified(spec, variables).residual));

/** Why what stayed in the residual stayed: each residue's reason and class. */
const reasons = (spec: unknown, variables: Variables = {}): string[][] =>
  simplified(spec, variables).residues.map((residue) => [residue.reason, residue.category]);

/** Whether two values are one Python value of one type: bytes by their contents, a complex number by its parts. */
const pyEqual = (a: unknown, b: unknown): boolean => Repr.typeName(a) === Repr.typeName(b) && (a instanceof Uint8Array
  ? F.sameNative(a, b) : a instanceof PD.ComplexValue ? Object.is(a.real, (b as any).real) && Object.is(a.imag, (b as any).imag) : a === b);

/** Each (expression, variables, value) folds to a literal root of that value, of its type. */
function folds(rows: [unknown, Variables, unknown][]): void {
  for (const [spec, variables, expected] of rows) {
    const found = value(spec, variables);
    assert(pyEqual(found, expected), `${source(spec)}: ${Repr.repr(found)}, not ${Repr.repr(expected)}`);
  }
}

/** Each (expression, variables, residual) simplifies to that residual, its root no literal and nothing named: a value
 * with no literal, a display, or a term whose subterms simplified. */
function stays(rows: [unknown, Variables, string][]): void {
  for (const [spec, variables, expected] of rows) {
    const result = simplified(spec, variables);
    const shown = PText.ToText(term(result.residual));
    assert(!result.literal && result.residues.length === 0 && shown === expected, `${source(spec)}: ${shown}, not ${expected}`);
  }
}

/** Each (expression, variables, reason, class) leaves one subterm in the residual, named by that reason and class. */
function named(rows: [unknown, Variables, string, string][]): void {
  for (const [spec, variables, reason, category] of rows) {
    const found = reasons(spec, variables);
    assert(found.length === 1 && found[0]![0] === reason && found[0]![1] === category, `${source(spec)}: ${found}`);
  }
}

/** Each (expression, variables, error, message) raises as Python raises it. */
function fails(rows: [unknown, Variables, Function, string][]): void {
  for (const [spec, variables, error, message] of rows) raises(error, () => simplified(spec, variables), message);
}

// the builders of the terms the cases write
const [n, kw, st, fv, comp] = [P.name, P.keyword, P.starred, P.formattedvalue, P.comprehension];
const [x, y, z, v, m] = [n("x"), n("y"), n("z"), n("v"), n("m")];
const [inf, nan] = [Infinity, NaN];
const s = (spec: unknown) => P.call("str", spec); // `str(spec)`, which shows what equality cannot (a NaN, a sign)
const cx = (re: number, im: number) => new PD.ComplexValue(re, im); // a complex number, which JavaScript has not
const [c1, c2, one] = [P.constant(cx(0, 1)), P.binop("+", 1n, P.constant(cx(0, 2))), P.constant(cx(1, 1))]; // 1j, 1 + 2j, (1+1j)

/** The simplification of the term `spec` writes, in a scope binding `variables`, with the options of simplification
 * `options` (PYS-15 to PYS-19). */
const under = (options: Variables) => (spec: unknown, variables: Variables = {}) =>
  PV.OfAny(term(spec), new PV.Scope(variables, { options }));

/** The message of the ValueError `run` throws. */
function refusal(run: () => unknown): string {
  try {
    run();
  } catch (error) {
    if (error instanceof ValueError) return error.message;
    throw error;
  }
  throw new Error("AssertionError: no ValueError");
}

<a id="PYS-01"></a>
## PYS-01 · Constants: `None`, `...` and complex numbers fold, and a value is written back as its literal, or as its display

In [ ]:
store.transaction(() => {
  // `None`, `...` and a complex number are Python3's constants: each folds to its value
  folds([
    [P.constant(null), {}, null], [P.unaryop("not", P.constant(null)), {}, true], [s(P.constant(P.ELLIPSIS)), {}, "Ellipsis"],
    [P.constant(cx(0, 2)), {}, cx(0, 2)], [(u: any) => u[P.CONSTANT]((k: any) => k.complex((c: any) => c.imag(2.0))), {}, cx(0, 2)],
    [s(P.constant(cx(0, 1.5))), {}, "1.5j"], [s(P.constant(cx(0, 2))), {}, "2j"],
  ]);
  assert(value(P.constant(P.ELLIPSIS)) === P.ELLIPSIS && simplified(P.constant(null)).literal);
  fails([
    [(u: any) => u[P.CONSTANT]((r: any) => r), {}, ValueError, "a constant needs a singleton or a complex number"],
    [P.binop("+", P.constant(P.ELLIPSIS), 1n), {}, TypeError, "unsupported operand type(s) for +: 'ellipsis' and 'int'"],
  ]);
  // A value is written back as its literal, a constant's among them: a literal root
  for (const [bound, written] of [[null, "None"], [P.ELLIPSIS, "..."], [cx(1, 2), "(1+2j)"], [cx(-0, -2), "(-0-2j)"], [-1.5, "-1.5"],
    [-3n, "-3"], [true, "True"], ["a", "'a'"], [new Uint8Array([122]), "b'z'"], [inf, "float('inf')"], [nan, "float('nan')"]] as [unknown, string][]) {
    assert(text(x, { x: bound }) === written && simplified(x, { x: bound }).literal, written);
  }
  // A list, a tuple, a dict or a set of values that have literals is written back as its display, a term, not a
  // literal; a value with no literal keeps its term
  stays([
    [P.tuple_(1n, P.binop("+", 1n, 1n)), {}, "(1, 2)"], [P.tuple_(1n), {}, "(1,)"], [x, { x: new Tuple([1n, null, P.ELLIPSIS]) }, "(1, None, ...)"],
    [x, { x: new Map([[new Tuple([1n, 2n]), [3n, new Tuple([4n])]]]) }, "{(1, 2): [3, (4,)]}"], [x, { x: [] }, "[]"], [x, { x: new Map() }, "{}"],
    [x, { x: new Set([2n, 1n]) }, "{1, 2}"], [x, { x: new Set() }, "x"], // a non-empty set's display, its items in order (PYS-13)
  ]);
  raises(NotEvaluated, () => simplified(P.lambda_(1n)).evaluate(), "the root is not a closed constructor expression: its value has no closed constructor expression");
});

<a id="PYS-02"></a>
## PYS-02 · Complex arithmetic as CPython 3.14 computes it: a real operand not made complex, infinities recovered, integer powers, `abs` correctly rounded; a power that needs the platform's functions stays, named

In [ ]:
store.transaction(() => {
  folds([
    [c2, {}, cx(1, 2)], [P.binop("*", c1, c1), {}, cx(-1, 0)], [P.binop("-", 1.5, c2), {}, cx(0.5, -2)],
    [P.binop("/", c2, P.binop("-", 3n, P.constant(cx(0, 4)))), {}, cx(-0.2, 0.4)], [P.binop("/", 1n, c2), {}, cx(0.2, -0.4)],
    [P.binop("**", P.binop("+", 1n, c1), -2n), {}, cx(0, -0.5)], [P.binop("**", P.binop("+", 1n, c1), 3n), {}, cx(-2, 2)],
    [P.unaryop("not", P.constant(cx(0, 0))), {}, true], [P.attribute(c2, "imag"), {}, 2.0],
    [P.compare("==", c1, P.binop("*", c1, 1n)), {}, true], [P.compare("==", P.binop("+", 2n, P.constant(cx(0, 0))), 2n), {}, true],
    [P.compare("==", 2n, P.binop("+", 2n, P.constant(cx(0, 0)))), {}, true], [s(P.binop("-", 1n, P.constant(cx(0, 2)))), {}, "(1-2j)"],
    [s(P.binop("+", P.constant(cx(1, -0)), 1.0)), {}, "(2-0j)"], // a real is not made complex (3.14)
    [s(P.binop("+", c1, 1.5)), {}, "(1.5+1j)"], [s(P.binop("-", c1, 1.5)), {}, "(-1.5+1j)"], [s(P.binop("*", 2.0, c1)), {}, "2j"],
    [s(P.binop("/", c1, 2.0)), {}, "0.5j"], [s(P.unaryop("-", c2)), {}, "(-1-2j)"], [s(P.unaryop("+", c1)), {}, "1j"],
    [s(P.binop("+", c1, c1)), {}, "2j"], [s(P.binop("-", c1, c1)), {}, "0j"], [s(P.binop("/", 1n, P.constant(cx(2, 1)))), {}, "(0.4-0.2j)"],
    [s(P.binop("/", one, P.constant(cx(4, 1)))), {}, "(0.29411764705882354+0.17647058823529413j)"], // Smith's algorithm
    [s(P.binop("**", P.constant(cx(0, 0)), 0.5)), {}, "0j"],
  ]);
  // Infinities and NaNs in products and quotients, recovered as C11's Annex G does
  folds([
    [s(P.binop("*", x, y)), { x: cx(inf, inf), y: cx(1, 0) }, "(inf+infj)"], [s(P.binop("*", y, x)), { x: cx(inf, inf), y: cx(1, 0) }, "(inf+infj)"],
    [s(P.binop("*", x, y)), { x: cx(nan, 1e300), y: cx(1e300, 1e300) }, "(-inf+infj)"], [s(P.binop("*", x, one)), { x: cx(nan, nan) }, "(nan+nanj)"],
    [s(P.binop("/", one, x)), { x: cx(nan, 1) }, "(nan+nanj)"], [s(P.binop("/", x, one)), { x: cx(inf, nan) }, "(inf-infj)"],
    [s(P.binop("/", one, x)), { x: cx(inf, inf) }, "0j"], [s(P.binop("/", 1n, x)), { x: cx(nan, 1) }, "(nan+nanj)"],
    [s(P.binop("/", x, one)), { x: cx(-inf, nan) }, "(-inf+infj)"], [s(P.binop("/", x, one)), { x: cx(nan, inf) }, "(inf+infj)"],
    [s(P.binop("/", one, x)), { x: cx(inf, nan) }, "0j"], [s(P.binop("/", one, x)), { x: cx(nan, inf) }, "-0j"],
  ]);
  // `abs` of a complex number, correctly rounded in both languages
  folds([
    [P.call("abs", P.binop("+", 3n, P.constant(cx(0, 4)))), {}, 5.0], [P.call("abs", P.constant(cx(0, 3))), {}, 3.0],
    [P.call("abs", P.binop("+", x, c1)), { x: inf }, inf], [s(P.call("abs", P.binop("+", x, c1))), { x: nan }, "nan"],
    [P.call("abs", x), { x: cx(5e-324, 5e-324) }, 5e-324],
  ]);
  // A complex power whose exponent is no integer, and a real power whose value is complex, need the platform's `pow`,
  // `atan2`, `cos` and `sin`: each stays, named
  named([
    [P.binop("**", c1, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"],
    [P.binop("**", -8n, 0.5), {}, "Python3 binop '**' of a complex power is not simplified", "a"],
  ]);
  fails([
    [P.binop("/", 1n, P.constant(cx(0, 0))), {}, ZeroDivisionError, "division by zero"],
    [P.binop("/", c1, 0.0), {}, ZeroDivisionError, "division by zero"], [P.binop("/", c1, P.constant(cx(0, 0))), {}, ZeroDivisionError, "division by zero"],
    [P.binop("**", P.constant(cx(0, 0)), -1n), {}, ZeroDivisionError, "zero to a negative or complex power"],
    [P.binop("**", P.constant(cx(0, 0)), c1), {}, ZeroDivisionError, "zero to a negative or complex power"],
    [P.binop("**", P.constant(cx(1e200, 1e200)), 2n), {}, OverflowError, "complex exponentiation"],
    [P.call("abs", P.binop("+", 1.5e308, P.constant(cx(0, 1.5e308)))), {}, OverflowError, "absolute value too large"],
    [P.compare("<", c1, c1), {}, TypeError, "'<' not supported between instances of 'complex' and 'complex'"],
    [P.binop("//", c1, 2n), {}, TypeError, "unsupported operand type(s) for //: 'complex' and 'int'"],
  ]);
});

<a id="PYS-03"></a>
## PYS-03 · Parentheses, adjacent strings and named expressions

In [ ]:
store.transaction(() => {
  folds([
    [P.binop("*", P.parenthesized(P.binop("+", 1n, 2n)), 3n), {}, 9n], [P.concatenatedstring("a", "b"), {}, "ab"],
    [P.concatenatedstring(new Uint8Array([97]), new Uint8Array([98])), {}, new Uint8Array([97, 98])], [P.binop("+", P.namedexpr("y", 2n), 1n), {}, 3n],
    [P.namedexpr("y", x), { x: 1.5 }, 1.5],
  ]);
  stays([[P.binop("*", P.parenthesized(P.binop("+", x, P.binop("+", 1n, 2n))), 3n), {}, "(x + 3) * 3"]]);
  fails([
    [P.concatenatedstring(new Uint8Array([97]), "b"), {}, TypeError, "cannot mix bytes and nonbytes literals"],
    [P.binop("+", new Uint8Array([97]), "b"), {}, TypeError, "can't concat str to bytes"],
  ]);
});

<a id="PYS-04"></a>
## PYS-04 · f-strings: conversions, debug fields, nested format specs and escapes; a residual's fields respelled; a field whose text is an implementation's stays, named

In [ ]:
store.transaction(() => {
  folds([
    [P.joinedstr(["x=", fv(x), "!"]), { x: 1.5 }, "x=1.5!"], [P.joinedstr([fv(x, "r", [">8"])]), { x: "a" }, "     'a'"],
    [P.joinedstr([fv(x, null, null, true, "x=")]), { x: new Tuple([1n, null]) }, "x=(1, None)"],
    [P.joinedstr([fv(x, "a")]), { x: "é" }, "'\\xe9'"], [P.joinedstr([fv(x, "a")]), { x: "\u2028\u{1F600}" }, "'\\u2028\\U0001f600'"],
    [P.joinedstr([fv(x, "a")]), { x: "α" }, "'\\u03b1'"],
    [P.joinedstr([fv(x, "s")]), { x: [1.0, null] }, "[1.0, None]"],
    [P.joinedstr([fv(x, null, [fv(n("w")), ".", fv(n("p")), "f"])]), { x: 3.14159, w: 8n, p: 3n }, "   3.142"],
    [P.joinedstr(["{{", fv(1n), "}}\\n\\x41\\u00e9\\101\\q"]), {}, "{1}\nAéA\\q"], [P.joinedstr(["\\n"], "rf"), {}, "\\n"],
    [P.joinedstr([fv(x, "r")]), { x: new Map([["a", new Tuple([1n])]]) }, "{'a': (1,)}"], [P.joinedstr([fv(P.slice_(1n, 2n), "r")]), {}, "slice(1, 2, None)"],
    [P.joinedstr([]), {}, ""], [P.joinedstr(["a", fv(1n)], null as any), {}, "a1"], [P.joinedstr([fv(x, null, [])]), { x: 1n }, "1"],
    [P.joinedstr([fv(x, null, null, true)]), { x: 1n }, "1"],
  ]);
  // A field whose term simplified loses its text as written: a debug field's label is written before it, with `!r` unless
  // it has a conversion or a format spec
  stays([
    [P.joinedstr([fv(P.binop("+", x, P.binop("+", 1n, 2n)), null, null, true, "x + (1+2)=")]), {}, "f'x + (1+2)={x + 3!r}'"],
    [P.joinedstr([fv(P.binop("+", x, P.binop("+", 1n, 2n)), null, [">5"], true, "x=")]), {}, "f'x={x + 3:>5}'"],
    [P.joinedstr([fv(P.binop("+", x, P.binop("+", 1n, 2n)), null, [], false, "x + (1+2)")]), {}, "f'{x + 3:}'"],
    [P.joinedstr([fv(P.binop("+", x, P.binop("+", 1n, 2n)), null, null, false, "x + (1+2)")]), {}, "f'{x + 3}'"],
    [P.let_("y", 2n, P.joinedstr([fv(x), fv(y)])), {}, "f'{x}{2}'"],
  ]);
  named([ // a set's text is its hash table's order (`\N{...}`: PYS-14)
    [P.joinedstr([fv(x)]), { x: new Set([1n]) }, "Python3 joinedstr's field of set is not simplified", "a"],
    [P.joinedstr([fv(x, "r")]), { x: new Set([1n]) }, "Python3 joinedstr's field of set is not simplified", "a"],
  ]);
});

<a id="PYS-05"></a>
## PYS-05 · `format()`'s mini-language for a str, an int, a bool, a float and a complex number, as CPython rounds and composes them; a locale's spec stays, named

In [ ]:
store.transaction(() => {
  folds(([
    [1234n, "08,", "0,001,234"], [-1n, "06", "-00001"], [1n, "=+6", "+    1"], [255n, "#010_x", "0x000_00ff"], [65n, ">3c", "  A"],
    [12345678n, "_b", "1011_1100_0110_0001_0100_1110"], [true, ">5", "    1"], [255n, "X", "FF"], [5n, "+", "+5"], [5n, " ", " 5"],
    [2n ** 60n, ".3e", "1.153e+18"], [12345.678, ",.2f", "12,345.68"], [2.675, ".2f", "2.67"], [-0.0, "z.1f", "0.0"],
    [1.5, "#.0f", "2."], [2.5, ".0f", "2"], [1234.5, ".3", "1.23e+03"], [1.0, ".3", "1.0"], [0.5, "%", "50.000000%"],
    [123456789.0, "g", "1.23457e+08"], [100.0, "#g", "100.000"], [1e-5, ".3e", "1.000e-05"], [1e-5, "E", "1.000000E-05"],
    [0.0, "e", "0.000000e+00"], [1e23, ".3e", "1.000e+23"], [9.9999, ".2e", "1.00e+01"], [1.5, "e", "1.500000e+00"],
    [1.5, ">6", "   1.5"], [1.5, " .1f", " 1.5"], [1e16, "", "1e+16"], [-inf, "E", "-INF"], [nan, "f", "nan"],
    [inf, "+f", "+inf"], [nan, "+.1f", "+nan"], ["ab", "^6", "  ab  "], ["ab", "0>5", "000ab"], ["héllo", ".2", "hé"],
    ["a", "*<3", "a**"]] as [unknown, string, string][]).map(([bound, spec, expected]) =>
    [P.joinedstr([fv(x, null, [spec])]), { x: bound }, expected] as [unknown, Variables, unknown]));
  // A complex number: each part a float's, the imaginary part signed, then `j`; with no type, a repr's digits (or `g`'s,
  // with a precision), in parentheses unless the real part is +0, which is left out; then padded as a whole
  folds(([
    [cx(1, 2), ">12", "      (1+2j)"], [cx(1.23456, 1), ".3", "(1.23+1j)"], [cx(0, 2), "+", "+2j"], [cx(1, 1), "#", "(1.+1.j)"],
    [cx(1e16, 1), "#", "(1.e+16+1.j)"], [cx(1234.5, 1), "_.2f", "1_234.50+1.00j"], [cx(1, 2), "E", "1.000000E+00+2.000000E+00j"],
    [cx(inf, nan), "F", "INF+NANj"], [cx(-0, -0), "z.1f", "0.0+0.0j"], [cx(1, 2), "-^15", "----(1+2j)-----"],
    [cx(1, 2), "*<9", "(1+2j)***"], [cx(1, 2), " ", "( 1+2j)"], [cx(-1.5, -2.5), "g", "-1.5-2.5j"]] as [unknown, string, string][]).map(([bound, spec, expected]) =>
    [P.joinedstr([fv(x, null, [spec])]), { x: bound }, expected] as [unknown, Variables, unknown]));
  named([ // a locale's; a spec Python refuses (zero padding for a complex number); a spec of a value with no format of its own
    [P.joinedstr([fv(x, null, ["n"])]), { x: 1n }, "Python3 joinedstr's format spec 'n' of int is not simplified", "a"],
    [P.joinedstr([fv(c1, null, ["n"])]), {}, "Python3 joinedstr's format spec 'n' of complex is not simplified", "a"],
    [P.joinedstr([fv(c1, null, ["05"])]), {}, "Python3 joinedstr's format spec '05' of complex is not simplified", "a"],
    [P.joinedstr([fv(x, null, [">5"])]), { x: new Set([1n]) }, "Python3 joinedstr's format of set is not simplified", "a"],
  ]);
  fails([
    [P.joinedstr([fv(P.constant(null), null, [">5"])]), {}, TypeError, "unsupported format string passed to NoneType.__format__"],
    [P.joinedstr([fv(x, null, ["c"])]), { x: 2n ** 70n }, OverflowError, "Python int too large to convert to C long"],
    [P.joinedstr([fv(x, null, ["c"])]), { x: -1n }, OverflowError, "%c arg not in range(0x110000)"],
  ]);
});

<a id="PYS-06"></a>
## PYS-06 · t-strings: a template, as Python 3.14's `string.templatelib` holds it, with no literal; its residual kept as written

In [ ]:
store.transaction(() => {
  const template = P.templatestr(["a", P.interpolation(x, null, null, true, "x = "), "b", P.interpolation(y, "r", [">3"])]);
  folds([
    [P.subscript(P.attribute(template, "strings"), 0n), { x: 1n, y: "s" }, "ax = "],
    [P.attribute(P.subscript(P.attribute(template, "interpolations"), 0n), "expression"), { x: 1n, y: "s" }, "x"],
    [P.attribute(P.subscript(P.attribute(template, "interpolations"), 0n), "conversion"), { x: 1n, y: "s" }, "r"],
    [P.attribute(P.subscript(P.attribute(template, "interpolations"), 1n), "format_spec"), { x: 1n, y: "s" }, ">3"],
    [P.subscript(P.attribute(template, "values"), 1n), { x: 1n, y: "s" }, "s"],
    [P.subscript(P.attribute(P.templatestr([]), "strings"), 0n), {}, ""],
    [P.subscript(P.attribute(P.templatestr(["a"], null as any), "strings"), 0n), {}, "a"],
    [P.subscript(P.attribute(P.templatestr([P.interpolation(x, null, null, true)]), "strings"), 0n), { x: 1n }, ""],
  ]);
  stays([[template, {}, "t'a{x = }b{y!r:>3}'"], [template, { x: 1n, y: 2n }, "t'a{x = }b{y!r:>3}'"]]); // its text is its value's
});

<a id="PYS-07"></a>
## PYS-07 · Displays and unpacking: tuples, sets by Python's equality, dicts and their `**` items, `*` in a display or a call, and their operators

In [ ]:
store.transaction(() => {
  folds([
    [P.compare("==", P.tuple_(1n, st(v)), P.tuple_(1n, 2n, 3n)), { v: [2n, 3n] }, true], [P.compare("==", P.tuple_(1n), P.list_(1n)), {}, false],
    [P.call("len", P.set_(1n, 1.0, true, 2n)), {}, 2n], [P.compare("in", 3n, P.set_(st(v))), { v: [3n] }, true],
    [P.compare("<", P.set_(1n), P.set_(1n, 2n)), {}, true], [P.call("len", P.binop("^", P.set_(1n, 2n), P.set_(2n, 3n))), {}, 2n],
    [P.subscript(P.dict_([null, m], ["b", 2n]), "a"), { m: { a: 1n } }, 1n], [P.call("max", st(v)), { v: [1n, 3n, 2n] }, 3n],
    [P.compare("<", P.tuple_(1n, 2n), P.tuple_(1n, 2n, 0n)), {}, true], [P.subscript(P.binop("+", P.tuple_(1n), P.tuple_(2n)), -1n), {}, 2n],
    [s(P.set_(1n)), {}, "{1}"], [s(P.call("set")), {}, "set()"], [P.unaryop("not", P.tuple_()), {}, true],
    [P.compare("==", P.set_(1n, 2n), P.set_(2n, 1n)), {}, true], [P.compare("==", P.dict_(["a", 1n]), m), { m: { a: 1n } }, true],
    [P.compare("==", P.dict_(["a", 1n]), P.dict_(["b", 1n])), {}, false], [P.compare("<", P.list_(1n, 2n), P.list_(1n, 3n)), {}, true],
    [P.compare("<=", P.set_(1n), P.set_(1n)), {}, true], [P.compare(">", P.set_(1n, 2n), P.set_(1n)), {}, true],
    [P.compare(">=", P.set_(1n), P.set_(1n, 2n)), {}, false], [P.call("len", P.binop("*", P.tuple_(1n, 2n), 2n)), {}, 4n],
    [P.call("len", P.binop("+", P.list_(1n), P.list_(2n))), {}, 2n], [P.call("len", P.binop("*", P.list_(1n), 2n)), {}, 2n],
    [P.subscript(P.dict_([P.tuple_(1n, 2n), 3n]), P.tuple_(1n, 2n)), {}, 3n], [P.call("len", P.binop("&", P.set_(1n, 2n), P.set_(2n, 3n))), {}, 1n],
    [P.call("len", P.binop("|", P.set_(1n), P.set_(2n))), {}, 2n], [P.subscript(P.binop("|", P.dict_(["a", 1n]), P.dict_(["b", 2n])), "b"), {}, 2n],
    [P.call("len", P.binop("-", P.set_(1n, 2n), P.set_(2n))), {}, 1n],
  ]);
  stays([
    [P.set_(P.binop("+", 1n, 1n), y), {}, "{2, y}"], [P.list_(st(v), 3n), { v: new Tuple([1n, 2n]) }, "[1, 2, 3]"], [P.list_(st(v)), {}, "[*v]"],
    [P.dict_([null, m], ["b", P.binop("+", 1n, 1n)]), { m: { a: 1n } }, "{'a': 1, 'b': 2}"], [P.dict_([null, m], ["b", 2n]), {}, "{**m, 'b': 2}"],
  ]);
  fails([
    [P.list_(st(v)), { v: 5n }, TypeError, "Value after * must be an iterable, not int"],
    [P.dict_([null, m]), { m: 1n }, TypeError, "'int' object is not a mapping"],
    [P.set_(v), { v: [1n] }, TypeError, "cannot use 'list' as a set element (unhashable type: 'list')"],
    [P.dict_([P.tuple_(1n, P.list_()), 2n]), {}, TypeError, "cannot use 'tuple' as a dict key (unhashable type: 'list')"],
    [P.binop("+", P.list_(1n), P.tuple_(2n)), {}, TypeError, "can only concatenate list (not \"tuple\") to list"],
    [P.list_(st(P.generatorexp(P.binop("/", 1n, n("p")), comp("p", v)))), { v: [0n] }, ZeroDivisionError, "division by zero"],
  ]);
});

<a id="PYS-08"></a>
## PYS-08 · Slices and subscripts by one: negative indices, steps and bounds clamped, as Python slices a sequence

In [ ]:
store.transaction(() => {
  folds([
    [P.subscript("abcde", P.slice_(1n, -1n)), {}, "bcd"], [P.subscript("abcde", P.slice_(null, null, -2n)), {}, "eca"],
    [P.subscript(new Uint8Array([97, 98, 99]), P.slice_(null, null, 2n)), {}, new Uint8Array([97, 99])],
    [P.call("len", P.subscript(v, P.slice_(5n, 1n, -2n))), { v: Array(6).fill(1n) }, 2n],
    [P.attribute(P.slice_(1n, 2n), "step"), {}, null], [P.call("len", P.subscript(P.list_(1n, 2n, 3n), P.slice_(-10n, null, -1n))), {}, 0n],
    [P.subscript(P.subscript(P.list_(1n, 2n, 3n), P.slice_(10n, null, -1n)), 0n), {}, 3n], [P.call("len", P.subscript(P.list_(1n, 2n, 3n), P.slice_(-10n))), {}, 3n],
    [P.subscript(P.subscript(P.tuple_(1n, 2n, 3n), P.slice_(1n)), 0n), {}, 2n], [P.call("len", P.subscript(P.list_(1n, 2n, 3n), P.slice_(10n))), {}, 0n],
  ]);
  stays([[P.subscript(v, P.slice_(1n)), { v: [1n, 2n, 3n] }, "[2, 3]"], [P.subscript(v, P.slice_(P.binop("+", 0n, 1n))), {}, "v[1:]"],
    [P.slice_(1n, 2n), {}, "1:2"]]); // a slice has no literal
  fails([
    [P.subscript(v, P.slice_(null, null, 0n)), { v: [1n] }, ValueError, "slice step cannot be zero"],
    [P.subscript(v, P.slice_(1.5)), { v: [1n] }, TypeError, "slice indices must be integers or None or have an __index__ method"],
    [P.subscript(P.tuple_(1n, 2n), 5n), {}, IndexError, "tuple index out of range"],
  ]);
});

<a id="PYS-09"></a>
## PYS-09 · Comprehensions: a list's, a set's and a dict's, of several clauses, over targets that unpack; a target that writes into data stays, named

In [ ]:
store.transaction(() => {
  folds([
    [P.call("sum", P.listcomp(P.binop("*", n("p"), n("q")), comp(P.tuple_(n("p"), n("q")), v))), { v: [new Tuple([1n, 2n]), new Tuple([3n, 4n])] }, 14n],
    [P.call("len", P.setcomp(P.binop("%", n("p"), 2n), comp("p", v))), { v: [1n, 2n, 3n] }, 2n],
    [P.subscript(P.dictcomp(n("p"), P.binop("*", n("p"), n("p")), comp("p", v, [P.compare(">", n("p"), 1n)])), 3n), { v: [1n, 2n, 3n] }, 9n],
    [P.call("sum", P.generatorexp(P.call("len", n("q")), comp(P.tuple_(n("p"), st(n("q"))), v))), { v: [[1n, 2n, 3n], [4n]] }, 2n],
    [P.call("sum", P.listcomp(n("p"), comp(P.parenthesized(n("p")), v))), { v: [1n, 2n] }, 3n],
    [P.call("sum", P.listcomp(n("q"), comp("p", v), comp("q", n("p")))), { v: [[1n, 2n], [3n]] }, 6n],
    [P.call("sum", P.generatorexp(st(n("p")), comp("p", v))), { v: [[1n, 2n], [3n]] }, 6n], // `(*p for p in v)`, Python 3.15
    [P.subscript(P.dictcomp(m, null, comp("m", v)), "b"), { v: [{ a: 1n }, { b: 2n }] }, 2n], // `{**m for m in v}`, Python 3.15
  ]);
  stays([
    [P.listcomp(P.binop("*", n("p"), 2n), comp("p", v)), { v: [1n, 2n] }, "[2, 4]"],
    [P.listcomp(P.binop("*", n("p"), P.binop("+", 1n, 1n)), comp("p", v)), {}, "[p * 2 for p in v]"],
    [P.dictcomp(n("p"), 0n, comp("p", "ab")), {}, "{'a': 0, 'b': 0}"], [P.setcomp(n("p"), comp("p", v)), { v: [1n] }, "{1}"],
  ]);
  named([ // a target that writes into data, and an `async for`, need a statement (d); two starred targets are no syntax (b)
    [P.listcomp(n("p"), comp(P.tuple_(n("p"), P.subscript(n("o"), 0n)), v)), { v: [1n] }, "Python3 subscript target is not simplified", "d"],
    [P.listcomp(n("p"), comp(P.attribute(n("o"), "a"), v)), { v: [1n] }, "Python3 attribute target is not simplified", "d"],
    [P.listcomp(n("p"), comp("p", v, [], true)), { v: [1n] }, "Python3 async comprehension is not simplified", "d"],
    [P.listcomp(n("p"), comp(P.tuple_(st(n("p")), st(n("q"))), v)), { v: [1n] }, "Python3 starred target is not simplified", "b"],
  ]);
  fails([
    [P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), { v: [1n] }, TypeError, "cannot unpack non-iterable int object"],
    [P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), { v: [[1n, 2n, 3n]] }, ValueError, "too many values to unpack (expected 2, got 3)"],
    [P.listcomp(n("p"), comp(P.tuple_(n("p"), n("q")), v)), { v: [[1n]] }, ValueError, "not enough values to unpack (expected 2, got 1)"],
    [P.listcomp(n("p"), comp(P.tuple_(n("p"), st(n("q")), n("r")), v)), { v: [[1n]] }, ValueError,
      "not enough values to unpack (expected at least 2, got 1)"],
  ]);
});

<a id="PYS-10"></a>
## PYS-10 · Lambdas and keywords: closures binding Python's parameters, with CPython's messages, and the builtins' keywords

In [ ]:
store.transaction(() => {
  folds([
    [P.call(P.lambda_(P.binop("+", n("a"), n("b")), ["a", P.arg("b", 10n)]), 1n), {}, 11n],
    [P.call(P.lambda_(n("k"), [], { kwonlyargs: ["k"] }), kw("k", 4n)), {}, 4n],
    [P.call(P.lambda_(P.call("len", n("r")), [], { vararg: "r" }), 1n, 2n, 3n), {}, 3n],
    [P.call(P.lambda_(P.subscript(n("o"), "z"), [], { kwarg: "o" }), kw(null, m)), { m: { z: 5n } }, 5n],
    [P.call(P.lambda_(n("a"), [], { posonlyargs: ["a"] }), 7n), {}, 7n], [P.call(P.lambda_(n("a"), ["a"]), st(v)), { v: [8n] }, 8n],
    [P.call("max", v, kw("key", P.lambda_(P.unaryop("-", x), ["x"]))), { v: [3n, 1n, 2n] }, 1n],
    [P.call("min", v, kw("default", 0n)), { v: [] }, 0n], [P.call("sum", v, kw("start", 10n)), { v: [1n, 2n] }, 13n],
  ]);
  // A closure has no literal: a lambda stays, its body simplified in the scope it is made in
  stays([
    [P.lambda_(P.binop("+", x, P.binop("+", 1n, 2n)), ["x"]), {}, "lambda x: x + 3"],
    [P.lambda_(P.binop("+", x, y), ["x"]), { y: 2n }, "lambda x: x + 2"], [P.call(P.lambda_(x, ["x"]), z), {}, "(lambda x: x)(z)"],
  ]);
  fails([
    [P.call(P.lambda_(n("a"), ["a"])), {}, TypeError, "<lambda>() missing 1 required positional argument: 'a'"],
    [P.call(P.lambda_(n("a"), ["a", "b", "c"])), {}, TypeError, "<lambda>() missing 3 required positional arguments: 'a', 'b', and 'c'"],
    [P.call(P.lambda_(n("a"), ["a", "b"]), 1n, 2n, 3n), {}, TypeError, "<lambda>() takes 2 positional arguments but 3 were given"],
    [P.call(P.lambda_(n("a"), ["a", P.arg("b", 1n)]), 1n, 2n, 3n), {}, TypeError, "<lambda>() takes from 1 to 2 positional arguments but 3 were given"],
    [P.call(P.lambda_(0n), 1n), {}, TypeError, "<lambda>() takes 0 positional arguments but 1 was given"],
    [P.call(P.lambda_(n("a"), ["a"], { kwonlyargs: ["k"] }), 1n, 2n, kw("k", 3n)), {}, TypeError,
      "<lambda>() takes 1 positional argument but 2 positional arguments (and 1 keyword-only argument) were given"],
    [P.call(P.lambda_(0n, [], { kwonlyargs: ["k", "j"] }), 1n, kw("k", 1n), kw("j", 2n)), {}, TypeError,
      "<lambda>() takes 0 positional arguments but 1 positional argument (and 2 keyword-only arguments) were given"],
    [P.call(P.lambda_(n("a"), ["a"]), 1n, kw("a", 2n)), {}, TypeError, "<lambda>() got multiple values for argument 'a'"],
    [P.call(P.lambda_(n("a"), ["a"]), kw("b", 2n)), {}, TypeError, "<lambda>() got an unexpected keyword argument 'b'"],
    [P.call(P.lambda_(n("k"), [], { kwonlyargs: ["k"] })), {}, TypeError, "<lambda>() missing 1 required keyword-only argument: 'k'"],
    [P.call(P.lambda_(n("a"), [], { posonlyargs: ["a"] }), kw("a", 1n)), {}, TypeError,
      "<lambda>() got some positional-only arguments passed as keyword arguments: 'a'"],
    [P.call(P.lambda_(n("a"), ["a"]), kw("a", 1n), kw(null, m)), { m: { a: 2n } }, TypeError, "<lambda>() got multiple values for keyword argument 'a'"],
    [P.call(P.lambda_(n("a"), ["a"]), kw(null, m)), { m: new Map([[1n, 5n]]) }, TypeError, "keywords must be strings"],
    [P.call(P.lambda_(n("a"), ["a"]), kw(null, m)), { m: [1n] }, TypeError, "<lambda>() argument after ** must be a mapping, not list"],
    [P.call(P.lambda_(n("a"), ["a"]), st(v)), { v: 5n }, TypeError, "<lambda>() argument after * must be an iterable, not int"],
    [P.call("max", v, kw("foo", 1n)), { v: [1n] }, TypeError, "max() got an unexpected keyword argument 'foo'"],
    [P.call("min", 1n, 2n, kw("default", 0n)), {}, TypeError, "Cannot specify a default for min() with multiple positional arguments"],
    [P.call("abs", kw("x", 1n)), {}, TypeError, "abs() takes no keyword arguments"],
  ]);
  // A function the scope provides takes keywords as Python's do; in TypeScript, none: a JavaScript function has no
  // keyword parameters (a deliberate difference)
  raises(TypeError, () => simplified(P.call(n("f"), kw("k", 1n)), { f: (value: unknown) => value }), "f() takes no keyword arguments");
});

<a id="PYS-11"></a>
## PYS-11 · `is` and `is not` by identity, and `@`: whether equal immutable values are one object Python leaves to its implementation, which stays, named

In [ ]:
store.transaction(() => {
  folds([
    [P.compare("is", x, P.constant(null)), { x: null }, true], [P.compare("is not", 1n, P.constant(null)), {}, true],
    [P.compare("is", x, x), { x: [1n] }, true], [P.compare("is", P.list_(1n), P.list_(1n)), {}, false],
    [P.compare("is", 1n, 1.0), {}, false], [P.compare("is", true, true), {}, true], [P.compare("is", 1n, 2n), {}, false],
    [P.compare("is", P.constant(P.ELLIPSIS), P.constant(P.ELLIPSIS)), {}, true], [P.compare("is not", x, y), { x: "a", y: "b" }, true],
  ]);
  named([
    [P.compare("is", 1n, 1n), {}, "Python3 compare 'is' of equal immutable values is not simplified", "b"],
    [P.compare("is", x, y), { x: nan, y: nan }, "Python3 compare 'is' of equal immutable values is not simplified", "b"],
    [P.compare("is not", x, y), { x: "a", y: "a" }, "Python3 compare 'is' of equal immutable values is not simplified", "b"],
  ]);
  // `@` calls an operand's `__matmul__`, which no builtin type has
  fails([[P.binop("@", x, y), { x: [1n], y: [2n] }, TypeError, "unsupported operand type(s) for @: 'list' and 'list'"]]);
});

<a id="PYS-12"></a>
## PYS-12 · What stays, by its class: `yield`, `yield from` and `await` (d), invalid syntax and a kind of the family's other dialects (b); `FOLDED`

In [ ]:
store.transaction(() => {
  named([
    [P.yield_(1n), {}, "Python3 yield is not simplified", "d"], [P.yieldfrom(v), { v: [1n] }, "Python3 yieldfrom is not simplified", "d"],
    [P.await_(x), { x: 1n }, "Python3 await is not simplified", "d"],
    [(u: any) => u.missing((r: any) => r), {}, "Python3 missing is not simplified", "b"],
  ]);
  // What stays keeps its subterms simplified, and what does not depend on it folds around it
  assert(text(P.binop("+", P.await_(P.binop("+", 1n, 2n)), 1n)) === "await 3 + 1");
  assert(JSON.stringify(reasons(P.binop("+", P.await_(P.binop("+", 1n, 2n)), 1n))) === JSON.stringify([["Python3 await is not simplified", "d"]]));
  assert(text(P.boolop("or", true, P.yield_(1n))) === "True");
  // A kind of Python Types, Python Patterns or Python RegExp is its own dialect's, of class (b): simplification across a
  // sub-dialect (mbse-basis CAP-299) is to reach it
  assert(["Python Types", "Python Patterns", "Python RegExp"].every((dialect) => PV.SIMPLIFIER.category(dialect, "typevar", "typevar") === "b"));
  assert(PV.SIMPLIFIER.category("Python3", "binop '**' of a complex power", "binop") === "a");
  assert(JSON.stringify([...PV.FOLDED].sort()) === JSON.stringify(["literal", "constant", "variable", "let", "list", "tuple", "set", "dict", "starred",
    "parenthesized", "concatenatedstring", "joinedstr", "templatestr", "attribute", "subscript", "slice", "call", "compare", "boolop", "binop",
    "unaryop", "ifexp", "namedexpr", "lambda", "listcomp", "setcomp", "dictcomp", "generatorexp", "import", "importfrom"].sort()));
  // What Python may never evaluate stays, though it would raise: a branch, by an unknown test; what follows an unknown
  // operand of `and` or `or`, or a chain's first comparison; a lambda's body
  const div = P.binop("/", 1n, 0n);
  for (const [spec, written] of [[P.ifexp(x, div, 2n), "1 / 0 if x else 2"], [P.boolop("and", x, div), "x and 1 / 0"],
    [P.compare("<", x, 1n, ["<", div]), "x < 1 < 1 / 0"], [P.lambda_(div), "lambda: 1 / 0"]] as [unknown, string][]) {
    assert(text(spec) === written && reasons(spec).length === 0);
  }
  raises(ZeroDivisionError, () => text(P.ifexp(true, div, 2n)), "division by zero"); // taken, it raises
});

<a id="PYS-13"></a>
## PYS-13 · Sets: a non-empty set written as its display, its items in an order both implementations compute; an empty set stays

In [ ]:
store.transaction(() => {
  // A non-empty set is written as its display, a closed constructor expression, its items in an order of their values
  // that both implementations compute, not its hash table's: None, `...`, real numbers (a NaN after every other), complex
  // numbers (by their real parts, then their imaginary ones), strs and bytes as Python orders them, then tuples, item by
  // item
  const bytes = (text: string) => new TextEncoder().encode(text);
  stays([
    [x, { x: new Set([10n, 9n, 2.5, true]) }, "{True, 2.5, 9, 10}"], [x, { x: new Set(["b", "a", "é", "B"]) }, "{'B', 'a', 'b', 'é'}"],
    [x, { x: new Set([null, P.ELLIPSIS, cx(0, 1), 1n, "a", bytes("z"), new Tuple([1n, 2n]), new Tuple([1n]), new Tuple([0n, "z"])]) },
      "{None, ..., 1, 1j, 'a', b'z', (0, 'z'), (1,), (1, 2)}"],
    [x, { x: new Set([nan, -inf, 1n]) }, "{-float('inf'), 1, float('nan')}"], [x, { x: new Set([bytes("b"), bytes("a")]) }, "{b'a', b'b'}"],
    [x, { x: new Set([cx(nan, 1), cx(1, 2), cx(0, 2), cx(0, 1)]) }, "{1j, 2j, (1+2j), (nan+1j)}"],
    [x, { x: new Set([new Tuple([null, 1n]), new Tuple([null, 0n]), new Tuple([P.ELLIPSIS, 0n])]) }, "{(None, 0), (None, 1), (..., 0)}"],
    [P.set_(3n, 1n, P.binop("+", 1n, 1n)), {}, "{1, 2, 3}"], [P.setcomp(P.binop("%", n("p"), 3n), comp("p", v)), { v: [5n, 4n] }, "{1, 2}"],
    [x, { x: [1n, new Set([2n])] }, "[1, {2}]"], [x, { x: new Map([["k", new Set([2n, 1n])]]) }, "{'k': {1, 2}}"],
  ]);
  // Its root a closed constructor expression, not a literal, the simplification evaluated the set
  assert(simplified(x, { x: new Set([2n, 1n]) }).constructed && !simplified(x, { x: new Set([2n, 1n]) }).literal);
  const evaluated = value(P.set_(2n, 1n));
  assert(evaluated instanceof Set && evaluated.size === 2 && evaluated.has(1n) && evaluated.has(2n));
  // The framework's `known`, which a dialect's `simplify` hook calls with a value it computed, writes a set's display too
  const known = PV.SIMPLIFIER.known(term(x), new Set([2n, 1n]));
  assert(known.constructed && !known.literal && PText.ToText(term(known.expression)) === "{1, 2}");
  // An empty set has no display (`{}` is a dict, and `set()` calls a name), nor has a set of an item with none: each stays
  stays([[x, { x: new Set() }, "x"], [P.call("set"), {}, "set()"], [x, { x: [new Set()] }, "x"],
    [P.set_(P.lambda_(1n)), {}, "{lambda: 1}"]]);
  raises(NotEvaluated, () => simplified(P.binop("-", P.set_(1n), P.set_(1n))).evaluate(), // its value an empty set
    "the root is not a closed constructor expression: its value has no closed constructor expression");
});

<a id="PYS-14"></a>
## PYS-14 · `\N{...}` by Unicode's character names and aliases, as CPython 3.14 resolves them, kept as data for TypeScript; a malformed escape and an unknown name stay, named (b)

In [ ]:
store.transaction(() => {
  // `\N{...}` names a character by a name or an alias of Unicode's, its case ignored, as CPython 3.14's compiler reads it:
  // Hangul's syllables and the ideographs' names by rule, but no named sequence
  folds([
    [P.joinedstr(["\\N{BULLET}"]), {}, "\u2022"], [P.joinedstr(["\\N{bullet}\\N{Latin Small Letter A}"]), {}, "\u2022a"],
    [P.joinedstr(["\\N{LF}\\N{LINE FEED}\\N{BYTE ORDER MARK}"]), {}, "\n\n\ufeff"], // aliases
    [P.joinedstr(["\\N{HANGUL SYLLABLE GAG}\\N{hangul syllable a}"]), {}, "\uac01\uc544"],
    [P.joinedstr(["\\N{CJK UNIFIED IDEOGRAPH-4E00}\\N{tangut ideograph-17000}\\N{CJK COMPATIBILITY IDEOGRAPH-2F800}"]), {},
      "\u4e00\u{17000}\u{2f800}"],
    [P.joinedstr(["a\\N{BULLET}}}b", fv(1n)]), {}, "a\u2022}b1"], [P.joinedstr(["\\N{BULLET}"], "rf"), {}, "\\N{BULLET}"],
    [P.call("len", P.joinedstr(["\\N{ARABIC LETTER TEH}\\N{ARABIC LETTER TEH MARBUTA}\\N{SEGMENTED DIGIT NINE}"])), {}, 3n],
  ]);
  assert(text(P.joinedstr(["\\N{BULLET}"])) === "'\u2022'" && simplified(P.joinedstr(["\\N{BULLET}"])).literal);
  // What Python refuses to compile, invalid syntax (b): an unknown name (a named sequence's, a non-ASCII one, an
  // ideograph's beyond its range or not in four uppercase hexadecimal digits at least), and a malformed escape
  named(["\\N{NO SUCH NAME}", "\\N{LATIN CAPITAL LETTER A WITH MACRON AND GRAVE}", "\\N{\u00e9}", "\\N{CJK UNIFIED IDEOGRAPH-04E00}",
    "\\N{CJK UNIFIED IDEOGRAPH-4E0G}", "\\N{CJK UNIFIED IDEOGRAPH-3300}", "\\N{CJK UNIFIED IDEOGRAPH-A000}"].map((spelling) =>
    [P.joinedstr([spelling]), {}, "Python3 joinedstr's unknown Unicode character name is not simplified", "b"] as [unknown, Variables, string, string]));
  named(["\\N", "\\NBULLET", "\\N{}", "\\N{BULLET"].map((spelling) =>
    [P.joinedstr([spelling]), {}, "Python3 joinedstr's malformed \\N character escape is not simplified", "b"] as [unknown, Variables, string, string]));
  // The names are kept as data, `grammars/python3-unicode-names.json`, which TypeScript reads, Python reading its own
  // `unicodedata`: Unicode 16.0's, 45,786 names, and 19 ranges of the ideographs' names by rule
  const data = JSON.parse(readFileSync("../../grammars/python3-unicode-names.json", "utf8"));
  const held = new Map<string, number>();
  for (const alias of data.aliases as string[]) {
    const [code, name] = alias.split(";") as [string, string];
    held.set(name, parseInt(code, 16));
  }
  let point = parseInt(data.hangul.first, 16);
  for (const initial of data.hangul.initial) {
    for (const medial of data.hangul.medial) for (const final of data.hangul.final) held.set(data.hangul.prefix + initial + medial + final, point++);
  }
  let previous: string[] = [];
  point = -1;
  for (const run of data.characters as string[]) { // a run of names that differ in their last word alone
    const [delta, shared, rest] = run.split(";") as [string, string, string];
    const [first, ...lasts] = rest.split("|") as [string, ...string[]];
    const words = [...previous.slice(0, Number(shared)), ...first.split(" ")];
    point += delta === "" ? 1 : parseInt(delta, 16);
    held.set(words.join(" "), point);
    for (const last of lasts) {
      words[words.length - 1] = last;
      held.set(words.join(" "), ++point);
    }
    previous = words;
  }
  assert(data.unicode === "16.0.0" && held.size === 45786 && data.ranges.length === 19);
});

<a id="PYS-15"></a>
## PYS-15 · A format spec Python refuses stays, named; the scope's option `specs` "raise" raises Python's `ValueError`, with CPython's message

In [ ]:
store.transaction(() => {
  // The options of the scope that take a string, each with its values, its default first; the simplifier refuses, before
  // it simplifies, a value an option does not take
  assert(JSON.stringify(PV.OPTIONS) === JSON.stringify({ specs: ["stay", "raise"], identity: ["stay", "cpython"],
    math: ["stay", "host"], text: ["stay", "cpython"] }));
  raises(ValueError, () => under({ specs: "strict" })(x), "the option 'specs' is 'stay' or 'raise', not 'strict'");
  assert(PV.OfAny(term(x), { x: 1n }).literal && PV.SIMPLIFIER.run(term(1n)).literal); // a mapping, or none: no options
  // A spec Python refuses stays, named; under "raise" it throws Python's ValueError, with CPython's message, which this
  // implementation models (Python checks each message against its own `format`)
  const raising = under({ specs: "raise" });
  for (const [bound, spec, owner, message] of [
    [1n, "abc", "int", "Invalid format specifier 'abc' for object of type 'int'"],
    [1.5, ",_", "float", "Cannot specify both ',' and '_'."], [1.5, "_,", "float", "Cannot specify both ',' and '_'."],
    [1.5, ".,_f", "float", "Cannot specify both ',' and '_'."], [1.5, "._,f", "float", "Cannot specify both ',' and '_'."],
    [1n, ".", "int", "Format specifier missing precision"],
    [1n, "99999999999999999999", "int", "Too many decimal digits in format string"],
    [1.5, ".99999999999999999999f", "float", "Too many decimal digits in format string"],
    [1n, ",c", "int", "Cannot specify ',' with 'c'."], [1n, ",x", "int", "Cannot specify ',' with 'x'."],
    ["ab", ",", "str", "Cannot specify ',' with 's'."], [1n, ".2,n", "int", "Cannot specify ',' with 'n'."],
    [1n, "_\x01", "int", "Cannot specify '_' with '\\x1'."],
    [1n, "é", "int", "Unknown format code '\\xe9' for object of type 'int'"],
    [true, "s", "bool", "Unknown format code 's' for object of type 'bool'"],
    [1.5, "d", "float", "Unknown format code 'd' for object of type 'float'"],
    ["ab", "d", "str", "Unknown format code 'd' for object of type 'str'"],
    ["ab", "+s", "str", "Sign not allowed in string format specifier"], ["ab", " ", "str", "Space not allowed in string format specifier"],
    ["ab", "z", "str", "Negative zero coercion (z) not allowed in string format specifier"],
    ["ab", "#", "str", "Alternate form (#) not allowed in string format specifier"],
    ["ab", "=5", "str", "'=' alignment not allowed in string format specifier"],
    [1n, ".2d", "int", "Precision not allowed in integer format specifier"],
    [1n, "zd", "int", "Negative zero coercion (z) not allowed in integer format specifier"],
    [65n, "+c", "int", "Sign not allowed with integer format specifier 'c'"],
    [65n, "#c", "int", "Alternate form (#) not allowed with integer format specifier 'c'"],
    [1.5, ".2147483648f", "float", "precision too big"], [cx(1, 2), ".2147483648", "complex", "precision too big"],
    [cx(1, 2), "%", "complex", "Unknown format code '%' for object of type 'complex'"],
    [cx(1, 2), "05", "complex", "Zero padding is not allowed in complex format specifier"],
    [cx(1, 2), "=5", "complex", "'=' alignment flag is not allowed in complex format specifier"],
  ] as [unknown, string, string, string][]) {
    const field = P.joinedstr([fv(x, null, [spec])]);
    assert(refusal(() => raising(field, { x: bound })) === message, spec);
    const found = reasons(field, { x: bound });
    assert(found.length === 1 && found[0]![0] === `Python3 joinedstr's format spec ${Repr.repr(spec)} of ${owner} is not simplified`
      && found[0]![1] === "a", spec);
  }
  // A spec Python accepts but the simplifier does not format stays under "raise" too: a fraction's digits grouped (3.14),
  // a locale's (PYS-18); a width in another script's digits folds, and `#` writes a point before an exponent
  for (const spec of [".3,f", ".,", "n"]) {
    const found = raising(P.joinedstr([fv(x, null, [spec])]), { x: 1.5 }).residues.map((r) => r.reason);
    assert(found.length === 1 && found[0] === `Python3 joinedstr's format spec ${Repr.repr(spec)} of float is not simplified`, spec);
  }
  folds([[P.joinedstr([fv(x, null, ["٣"])]), { x: 1n }, "  1"], [P.joinedstr([fv(x, null, ["#"])]), { x: 1e16 }, "1.e+16"]]);
});

<a id="PYS-16"></a>
## PYS-16 · `is` between equal immutable values: the scope's option `identity` "cpython" answers as CPython does where the values decide it, for the values it keeps one object of; any other pair stays, named

In [ ]:
store.transaction(() => {
  // Under "cpython", CPython 3.14's answer where the values decide it: True for its small ints (-5 to 256) and the empty
  // tuple, str and bytes, which it keeps one object of, however they are made; a closure's scope keeps the options
  const cpython = under({ identity: "cpython" });
  for (const [spec, variables] of [[P.compare("is", 1n, 1n), {}], [P.compare("is", P.unaryop("-", 5n), x), { x: -5n }],
    [P.compare("is", 256n, x), { x: 256n }], [P.compare("is", P.tuple_(), P.tuple_()), {}],
    [P.compare("is", "", x), { x: "" }], [P.compare("is", new Uint8Array(), new Uint8Array()), {}],
    [P.call(P.lambda_(P.compare("is", n("a"), 1n), ["a"]), 1n), {}]] as [unknown, Variables][]) {
    assert(cpython(spec, variables).evaluate() === true, source(spec));
  }
  assert(cpython(P.compare("is not", x, y), { x: 7n, y: 7n }).evaluate() === false);
  // Whether any other pair of equal immutable values is one object depends on how they were made: it stays, named, as by
  // default (PYS-11)
  for (const [spec, variables] of [[P.compare("is", 257n, x), { x: 257n }], [P.compare("is", x, -6n), { x: -6n }],
    [P.compare("is", "a", "a"), {}], [P.compare("is", 1.0, 1.0), {}], [P.compare("is", P.tuple_(1n), P.tuple_(1n)), {}],
    [P.compare("is", x, y), { x: nan, y: nan }]] as [unknown, Variables][]) {
    const found = cpython(spec, variables).residues.map((r) => [r.reason, r.category]);
    assert(JSON.stringify(found) === JSON.stringify([["Python3 compare 'is' of equal immutable values is not simplified", "b"]]), source(spec));
  }
  raises(ValueError, () => under({ identity: "yes" })(x), "the option 'identity' is 'stay' or 'cpython', not 'yes'");
});

<a id="PYS-17"></a>
## PYS-17 · A complex power that needs the platform's functions: the scope's option `math` "host" folds it by the host's math library, as CPython's `_Py_c_pow` computes it

In [ ]:
store.transaction(() => {
  // Under "host", a complex power whose exponent is no integer, and a real power whose value is complex, fold by the host's
  // `hypot`, `pow`, `atan2`, `exp`, `log`, `cos` and `sin`, each host's last bit its own: the cases assert exact results
  const host = under({ math: "host" });
  for (const [spec, written] of [[s(P.binop("**", -1n, 0.5)), "(6.123233995736766e-17+1j)"],
    [s(P.binop("**", P.binop("+", 4n, P.constant(cx(0, 0))), 0.5)), "(2+0j)"],
    [s(P.binop("**", P.binop("+", 1n, P.constant(cx(0, 0))), c1)), "(1+0j)"]] as [unknown, string][]) {
    assert(host(spec).evaluate() === written, source(spec));
  }
  raises(OverflowError, () => host(P.binop("**", P.constant(cx(1e300, 1e300)), 2.5)), "complex exponentiation");
  // By default they stay, named (PYS-02); a negative infinity to a fractional power is a real infinity, as Python's is
  assert(under({})(P.binop("**", -1n, 0.5)).residues.map((r) => r.reason).join() === "Python3 binop '**' of a complex power is not simplified");
  folds([[P.binop("**", x, 0.5), { x: -inf }, inf]]);
  raises(ValueError, () => under({ math: "libm" })(x), "the option 'math' is 'stay' or 'host', not 'libm'");
});

<a id="PYS-18"></a>
## PYS-18 · A format spec a locale decides (`n`): the scope's option `locale`, a locale as `locale.localeconv()` gives one, formats it under that locale, as CPython does

In [ ]:
store.transaction(() => {
  // A locale is data: its decimal point, its thousands separator and its grouping, each group's size from the right, 0
  // repeating the last and 127 (C's CHAR_MAX) ending the grouping; an int is written as `d` writes it, a float and each
  // part of a complex number as `g` writes them, their integer digits grouped
  const de = { decimal_point: ",", thousands_sep: ".", grouping: [3, 0] };
  const c = { decimal_point: ".", thousands_sep: "", grouping: [] };
  const india = { decimal_point: ".", thousands_sep: ",", grouping: [3, 2, 0] };
  const once = { decimal_point: ".", thousands_sep: "'", grouping: [1, 127] };
  for (const [given, bound, spec, written] of [
    [de, 1234567n, "n", "1.234.567"], [de, 1234567.25, "n", "1,23457e+06"], [de, 12345.678, ".7n", "12.345,68"],
    [de, 1234n, "012n", "0.000.001.234"], [de, -1234.5, "+n", "-1.234,5"], [de, 1234.5, "*^11n", "**1.234,5**"],
    [de, 1234.5, "#.6n", "1.234,50"], [de, true, "n", "1"], [de, cx(1234.5, 2), "n", "1.234,5+2j"],
    [de, cx(1234.5, 2), ">14n", "    1.234,5+2j"], [de, 1234n, "<8n", "1.234   "], [de, -1234n, "*=9n", "-***1.234"],
    [c, 1234567n, "n", "1234567"], [c, 1234.5678, "n", "1234.57"], [c, -0, "zn", "0"], [c, inf, "08n", "00000inf"],
    [india, 123456789n, "n", "12,34,56,789"], [once, 12345n, "n", "1234'5"],
  ] as [Variables, unknown, string, string][]) {
    assert(under({ locale: given })(P.joinedstr([fv(x, null, [spec])]), { x: bound }).evaluate() === written, spec);
  }
  // Python checks the formatting against its own `format`, in the C locale and with `,` grouping by three
  // What Python refuses stays, named, under a locale too (PYS-15): a str's `n`, and `n` with `,`
  for (const [bound, spec, owner] of [["ab", "n", "str"], [1n, ",n", "int"]] as [unknown, string, string][]) {
    const found = under({ locale: de })(P.joinedstr([fv(x, null, [spec])]), { x: bound }).residues.map((r) => r.reason);
    assert(found.join() === `Python3 joinedstr's format spec ${Repr.repr(spec)} of ${owner} is not simplified`, spec);
  }
  for (const [value, message] of [
    ["de", "the option 'locale' is 'stay' or a mapping of 'decimal_point', 'thousands_sep' and 'grouping', not 'de'"],
    ...[{ decimal_point: "," }, { ...de, thousands_sep: 1n }, { ...de, grouping: "3" }, { ...de, grouping: ["3"] },
      { ...de, grouping: [1.5] }, { ...de, grouping: [-1] }, { ...de, grouping: [128] }].map((bad) =>
      [bad, "the option 'locale' maps 'decimal_point' and 'thousands_sep' to strs, and 'grouping' to a list of ints from 0 to 127"]),
  ] as [unknown, string][]) {
    assert(refusal(() => under({ locale: value })(x)) === message, String(message));
  }
});

<a id="PYS-19"></a>
## PYS-19 · A field whose text is the implementation's: the scope's option `text` "cpython" writes it as CPython writes it, where this implementation can

In [ ]:
store.transaction(() => {
  // Under "cpython", a template's text is its `repr`, as CPython 3.14 writes it, and a format spec of a value whose
  // `__format__` is `object`'s throws its TypeError, named as CPython names its type; by default they stay, named (PYS-04)
  const texts = under({ text: "cpython" });
  const tstring = P.templatestr(["a", P.interpolation(x, null, null, false, "x")]);
  const shown = "Template(strings=('a', ''), interpolations=(Interpolation(1, 'x', None, ''),))";
  for (const [spec, written] of [[P.joinedstr([fv(tstring, "r")]), shown], [P.joinedstr([fv(tstring)]), shown],
    [P.joinedstr([fv(P.subscript(P.attribute(tstring, "interpolations"), 0n), "a")]), "Interpolation(1, 'x', None, '')"]] as [unknown, string][]) {
    assert(texts(spec, { x: 1n }).evaluate() === written, source(spec));
  }
  named([[P.joinedstr([fv(tstring)]), { x: 1n }, "Python3 joinedstr's field of Template is not simplified", "a"]]);
  for (const [spec, variables, name] of [[P.joinedstr([fv(tstring, null, [">5"])]), { x: 1n }, "string.templatelib.Template"],
    [P.joinedstr([fv(P.subscript(P.attribute(tstring, "interpolations"), 0n), null, [">5"])]), { x: 1n }, "string.templatelib.Interpolation"],
    [P.joinedstr([fv(x, null, [">5"])]), { x: new Set([1n]) }, "set"],
    [P.joinedstr([fv(P.lambda_(1n), null, [">5"])]), {}, "function"],
    [P.joinedstr([fv(P.generatorexp(n("p"), comp("p", v)), null, [">5"])]), { v: [1n] }, "generator"]] as [unknown, Variables, string][]) {
    raises(TypeError, () => texts(spec, variables), `unsupported format string passed to ${name}.__format__`);
  }
  // A set's text, which is its hash table's order, and an object's, which is its address: no JavaScript value has them,
  // so they stay, named, where Python writes CPython's text (EQUIVALENCE); a host value's `__format__` is unknown, and
  // it stays too
  class Host {}
  for (const [spec, variables, reason] of [[P.joinedstr([fv(x)]), { x: new Set([2n, 1n]) }, "joinedstr's field of set"],
    [P.joinedstr([fv(P.lambda_(1n))]), {}, "joinedstr's field of function"],
    [P.joinedstr([fv(P.generatorexp(n("p"), comp("p", v)), "r")]), { v: [1n] }, "joinedstr's field of generator"],
    [P.joinedstr([fv(x, null, [">5"])]), { x: new Host() }, "joinedstr's format of Host"]] as [unknown, Variables, string][]) {
    assert(texts(spec, variables).residues.map((r) => r.reason).join() === `Python3 ${reason} is not simplified`, reason);
  }
  raises(ValueError, () => under({ text: "cp" })(x), "the option 'text' is 'stay' or 'cpython', not 'cp'");
});